# Predictive Maintenance Using NASA C-MAPSS

## Phase 1 — Data Ingestion

**Medallion Architecture: Bronze Layer**

Load the original NASA C-MAPSS FD001 training data, test data, and RUL labels into Delta tables while preserving the source data.

In [0]:
from pyspark.sql import functions as F

schema = "workspace.predictive_maintenance_rul"
source_path = "/Volumes/workspace/default/cmapss_data"

print("Project schema:", schema)
print("Source folder:", source_path)

Project schema: workspace.predictive_maintenance_rul
Source folder: /Volumes/workspace/default/cmapss_data


In [0]:
files = dbutils.fs.ls(source_path)

for file in files:
    print(file.name)

Damage Propagation Modeling.pdf
RUL_FD001.txt
RUL_FD002.txt
RUL_FD003.txt
RUL_FD004.txt
mlflow_tmp/
readme.txt
test_FD001.txt
test_FD002.txt
test_FD003.txt
test_FD004.txt
train_FD001.txt
train_FD002.txt
train_FD003.txt
train_FD004.txt
x.txt


In [0]:
# Bronze layer: load the original FD001 files
from pyspark.sql.types import StructType, StructField, DoubleType

# NASA FD001 contains 26 values per row:
# unit_number, cycle, 3 operating settings, 21 sensors
column_names = (
    ["unit_number", "cycle"]
    + [f"op_setting_{i}" for i in range(1, 4)]
    + [f"sensor_{i}" for i in range(1, 22)]
)

schema_fields = [
    StructField(name, DoubleType(), True)
    for name in column_names
]

# Engine and cycle identifiers will be converted to integers later.
raw_schema = StructType(schema_fields)

train_raw = (
    spark.read
    .option("delimiter", " ")
    .option("header", "false")
    .option("mode", "DROPMALFORMED")
    .schema(raw_schema)
    .csv(f"{source_path}/train_FD001.txt")
)

test_raw = (
    spark.read
    .option("delimiter", " ")
    .option("header", "false")
    .option("mode", "DROPMALFORMED")
    .schema(raw_schema)
    .csv(f"{source_path}/test_FD001.txt")
)

rul_raw = (
    spark.read
    .text(f"{source_path}/RUL_FD001.txt")
    .selectExpr("cast(trim(value) as int) as official_RUL")
)

# Save separate Bronze Delta tables
train_raw.write.format("delta").mode("overwrite").saveAsTable(
    f"{schema}.bronze_train_raw"
)

test_raw.write.format("delta").mode("overwrite").saveAsTable(
    f"{schema}.bronze_test_raw"
)

rul_raw.write.format("delta").mode("overwrite").saveAsTable(
    f"{schema}.bronze_test_rul_raw"
)

print("Bronze tables created successfully.")
print("Training rows:", spark.table(f"{schema}.bronze_train_raw").count())
print("Test rows:", spark.table(f"{schema}.bronze_test_raw").count())
print("Official RUL labels:", spark.table(f"{schema}.bronze_test_rul_raw").count())

Bronze tables created successfully.
Training rows: 0
Test rows: 0
Official RUL labels: 100


In [0]:
from pyspark.sql.types import (
    StructType, StructField, IntegerType, DoubleType
)

column_names = (
    ["unit_number", "cycle"]
    + [f"op_setting_{i}" for i in range(1, 4)]
    + [f"sensor_{i}" for i in range(1, 22)]
)

schema_fields = [
    StructField("unit_number", IntegerType(), True),
    StructField("cycle", IntegerType(), True),
] + [
    StructField(name, DoubleType(), True)
    for name in column_names[2:]
]

raw_schema = StructType(schema_fields)

def read_nasa_file(path):
    rows = (
        spark.read.text(path)
        .selectExpr("split(trim(value), '\\\\s+') as values")
    )

    return rows.selectExpr(
        *[
            f"cast(values[{i}] as "
            f"{'int' if i < 2 else 'double'}) as `{name}`"
            for i, name in enumerate(column_names)
        ]
    )

train_raw = read_nasa_file(f"{source_path}/train_FD001.txt")
test_raw = read_nasa_file(f"{source_path}/test_FD001.txt")

print("Training rows:", train_raw.count())
print("Test rows:", test_raw.count())

display(train_raw.limit(5))

Training rows: 20631
Test rows: 13096


unit_number,cycle,op_setting_1,op_setting_2,op_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,sensor_6,sensor_7,sensor_8,sensor_9,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
1,1,-7.0E-4,-4.0E-4,100.0,518.67,641.82,1589.7,1400.6,14.62,21.61,554.36,2388.06,9046.19,1.3,47.47,521.66,2388.02,8138.62,8.4195,0.03,392.0,2388.0,100.0,39.06,23.419
1,2,0.0019,-3.0E-4,100.0,518.67,642.15,1591.82,1403.14,14.62,21.61,553.75,2388.04,9044.07,1.3,47.49,522.28,2388.07,8131.49,8.4318,0.03,392.0,2388.0,100.0,39.0,23.4236
1,3,-0.0043,3.0E-4,100.0,518.67,642.35,1587.99,1404.2,14.62,21.61,554.26,2388.08,9052.94,1.3,47.27,522.42,2388.03,8133.23,8.4178,0.03,390.0,2388.0,100.0,38.95,23.3442
1,4,7.0E-4,0.0,100.0,518.67,642.35,1582.79,1401.87,14.62,21.61,554.45,2388.11,9049.48,1.3,47.13,522.86,2388.08,8133.83,8.3682,0.03,392.0,2388.0,100.0,38.88,23.3739
1,5,-0.0019,-2.0E-4,100.0,518.67,642.37,1582.85,1406.22,14.62,21.61,554.0,2388.06,9055.15,1.3,47.28,522.19,2388.04,8133.8,8.4294,0.03,393.0,2388.0,100.0,38.9,23.4044


In [0]:
# Remove only the new rebuild's Bronze tables
spark.sql(f"DROP TABLE IF EXISTS {schema}.bronze_train_raw")
spark.sql(f"DROP TABLE IF EXISTS {schema}.bronze_test_raw")

# Save the correctly parsed data as Delta tables
train_raw.write.format("delta").mode("overwrite").saveAsTable(
    f"{schema}.bronze_train_raw"
)

test_raw.write.format("delta").mode("overwrite").saveAsTable(
    f"{schema}.bronze_test_raw"
)

# Read the official RUL labels
test_rul_raw = (
    spark.read
    .text(f"{source_path}/RUL_FD001.txt")
    .selectExpr("cast(trim(value) as int) as official_rul")
)

spark.sql(f"DROP TABLE IF EXISTS {schema}.bronze_test_rul_raw")

test_rul_raw.write.format("delta").mode("overwrite").saveAsTable(
    f"{schema}.bronze_test_rul_raw"
)

print("All three Bronze tables saved successfully.")

All three Bronze tables saved successfully.


In [0]:
print("Training:", spark.table(f"{schema}.bronze_train_raw").count())
print("Test:", spark.table(f"{schema}.bronze_test_raw").count())
print("Official RUL labels:", spark.table(f"{schema}.bronze_test_rul_raw").count())

Training: 20631
Test: 13096
Official RUL labels: 100
